In [ ]:
# Load all metrics npz files from Results/{timestamp}.
# They are named {dataset}_{sampler}_{proposal}_{ss}_metrics.npz beside the
# {dataset}_{sampler}_{proposal}_{ss}.h5 they were evaluated from, where {ss}
# is ss_prop * 100 with any decimal point written as "p" (covtype_smc_DA_12p5).
# MH takes no subsample and is always tagged 100.

import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams.update({'text.usetex': False})
import matplotlib.patheffects as pe


In [ ]:


def parse_ss(tag):
    """The {ss} field of a filename back to ss_prop: '12p5' -> 0.125, '50' -> 0.5."""
    return float(tag.replace("p", ".")) / 100


def load_metrics_files(timestamp, results_root="Results"):
    """(smc, mcmc) lists of dicts, one per *_metrics.npz in the run directory."""
    results_path = os.path.join(results_root, timestamp)
    PROPOSALS = {"MH": "MH", "DA": "DA", "FlatHINTS": "Flat HINTS", "HINTS": "HINTS"}
    smc_samplers = []
    mcmc_samplers = []

    for filename in sorted(os.listdir(results_path)):
        if not filename.endswith("_metrics.npz"):
            continue

        # Split from the right: only the last three fields are in a fixed
        # position, so a dataset name with "_" in it still parses.
        stem = filename[:-len("_metrics.npz")]
        dataset, sampler, proposal, ss = stem.rsplit("_", 3)
        # MH runs on the full data -- None keeps it apart from the subsampled runs.
        ss_prop = None if proposal == "MH" else parse_ss(ss)

        file_path = os.path.join(results_path, filename)
        with np.load(file_path) as npz:
            # Read the arrays out now: np.load is lazy and holds the file open.
            data = {key: npz[key] for key in npz.files}

        sampler_dict = {
            "name": stem,
            "dataset": dataset,
            "ss_prop": ss_prop,
            "sampler": sampler,
            "proposal_mechanism": PROPOSALS.get(proposal, proposal),
            "path": file_path,
            "h5_path": os.path.join(results_path, stem + ".h5"),
            # Every stride-th stored record was evaluated (80 for smc_MH_100, 1
            # for the rest). data['iterations'] already holds the iteration /
            # step each evaluated point came from, so plot against that rather
            # than against the point's index.
            "stride": int(data["stride"]),
            "data": data,
        }

        if sampler == "smc":
            smc_samplers.append(sampler_dict)
        else:
            mcmc_samplers.append(sampler_dict)

    return smc_samplers, mcmc_samplers


In [ ]:
timestamp="susy_results"
smc_results, mcmc_results = load_metrics_files(timestamp)

In [ ]:
for experiment in smc_results:
    print(f"{experiment['proposal_mechanism']}, {experiment['ss_prop']}")

In [ ]:
# reorder the results to go in the order: DA, Flat HINTS, HINTS, MH and then for each of the first 3, order by ss_prop ascending
def reorder_results(results):
    proposal_order = ["DA", "Flat HINTS", "HINTS", "MH"]
    results.sort(key=lambda x: (proposal_order.index(x["proposal_mechanism"]), x["ss_prop"] if x["ss_prop"] is not None else float('inf')))
    return results

mcmc_results = reorder_results(mcmc_results)
smc_results = reorder_results(smc_results)

In [ ]:
for experiment in smc_results + mcmc_results:
    if experiment["proposal_mechanism"] == "HINTS":
        # The hierarchy design, e.g. ss_prop=0.125 -> [1, 2, 4, 8]: 1/ss_prop
        # blocks at the bottom level, halving until a single block at the top.
        ss_prop = experiment["ss_prop"]
        experiment["ss_prop"] = [2**i for i in range(int(np.log2(round(1 / ss_prop))) + 1)]
    elif experiment["proposal_mechanism"] == "Flat HINTS":
        experiment["ss_prop"] = int(round(1 / experiment["ss_prop"]))


In [ ]:
# Per-iteration runtime lives in the .h5 each metrics file was evaluated from:
# every Run_N group holds iter_time and cumulative_time, one entry per
# iteration (MCMC) or step (SMC). The clock times the sampler alone -- the
# time spent recording states is excluded.

import h5py


def record_positions(experiment):
    """Which stored records were evaluated: every stride-th one. state_index
    is indexed by record, while cumulative_time and ess are indexed by
    iteration -- the two only coincide when stride and store_every are 1, which
    the stride-80 smc_MH_100 run is not."""
    return np.arange(len(experiment['data']['iterations'])) * experiment['stride']


def load_n_iters(experiment):
    """Iterations (MCMC) / steps (SMC) the run actually took."""
    with h5py.File(experiment['h5_path'], "r") as f:
        return len(f["Run_0"]["cumulative_time"])


def load_runtime(experiment):
    """Cumulative sampling time in seconds, as an (n_runs, n_points) array
    lined up with the experiment's metric arrays."""
    h5_path = experiment['h5_path']
    iterations = experiment['data']['iterations']
    n_runs = int(experiment['data']['n_runs'])
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")), key=lambda k: int(k[len("Run_"):]))
        return np.stack([f[run]["cumulative_time"][()][iterations] for run in runs[:n_runs]])

def load_ess(experiment): # only for SMC experiments
    """Effective sample size (ESS) per iteration, as an (n_runs, n_points) array
    lined up with the experiment's metric arrays."""
    h5_path = experiment['h5_path']
    iterations = experiment['data']['iterations']
    n_runs = int(experiment['data']['n_runs'])
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")), key=lambda k: int(k[len("Run_"):]))
        return np.stack([f[run]["ess"][()][iterations] for run in runs[:n_runs]])

def load_tree_len(experiment, particles="mean"):
    """Tree size in nodes, as an (n_runs, n_points) array lined up with the
    experiment's metric arrays.

    state_tree_lengths holds one entry per *stored* tree, not per iteration:
    consecutive identical states are kept once, so a 10k-iteration run may
    store a couple of hundred trees. state_index is what maps an iteration (and,
    for SMC, a particle) onto a slot in that store -- indexing the store with
    the iteration numbers directly reads unrelated trees and runs off the end.

    MCMC carries one tree per iteration; SMC carries one per particle, so
    `particles` says how to reduce them: "mean", "max", or "all" to keep the
    full (n_runs, n_points, n_particles) array. All three agree on MCMC, where
    there is only one particle to reduce over.
    """
    reducers = {"mean": lambda a: a.mean(axis=1),
                "max": lambda a: a.max(axis=1),
                "all": lambda a: a}
    if particles not in reducers:
        raise ValueError(f"particles must be one of {sorted(reducers)}, got {particles!r}")
    reduce = reducers[particles]

    h5_path = experiment['h5_path']
    n_runs = int(experiment['data']['n_runs'])
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")), key=lambda k: int(k[len("Run_"):]))
        return np.stack([
            reduce(f[run]["state_tree_lengths"][()][f[run]["state_index"][()][record_positions(experiment)]])
            for run in runs[:n_runs]])


for experiment in smc_results + mcmc_results:
    experiment['time'] = load_runtime(experiment)
    experiment['tree_len'] = load_tree_len(experiment)
    experiment['n_iters'] = load_n_iters(experiment)

for experiment in smc_results:
    experiment['ess'] = load_ess(experiment)

In [ ]:
# plot mean and std of ess for all SMC experiments, each plot having a different proposal mechanism.
num_plots = len(smc_results)-1
ncols = 5
nrows = int(np.ceil(num_plots / ncols))
mh_tree_len_mean = np.mean([experiment['tree_len'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_tree_len_std = np.std([experiment['tree_len'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_time_mean = np.mean([experiment['time'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)

fig,axes=plt.subplots(nrows=nrows,ncols=ncols,figsize=(ncols*6,nrows*4),sharey=True)
for i, experiment in enumerate([e for e in smc_results if e['proposal_mechanism'] != 'MH']):
    ax = axes[i // ncols, i % ncols]
    tree_len_mean = np.mean(experiment['tree_len'], axis=0)
    tree_len_std = np.std(experiment['tree_len'], axis=0)
    time_mean = np.mean(experiment['time'], axis=0)
    ax.plot(time_mean, tree_len_mean, label='Mean Tree Length')
    ax.fill_between(time_mean, tree_len_mean - tree_len_std, tree_len_mean + tree_len_std, alpha=0.3, label='Std Dev')
    ax.plot(mh_time_mean, mh_tree_len_mean, label='MH Mean Tree Length', linestyle='--')
    ax.fill_between(mh_time_mean, mh_tree_len_mean - mh_tree_len_std, mh_tree_len_mean + mh_tree_len_std, alpha=0.3, label='MH Std Dev', linestyle='--')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Wall Clock Time (s)')
    ax.set_ylabel('Tree Length')
    # set xlim to be from 0 to the first iteration where tree_len == 50
    # ax.set_xlim([-10,160])
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Tree Length for SMC Experiments", fontsize=16,y=1.03)
plt.show()

In [ ]:
# plot mean and std of ess for all SMC experiments, each plot having a different proposal mechanism.
num_plots = len(smc_results)-1
ncols = 5
nrows = int(np.ceil(num_plots / ncols))
mh_tree_len_mean = np.mean([experiment['tree_len'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_tree_len_std = np.std([experiment['tree_len'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_time_mean = np.mean([experiment['time'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_iterations = [experiment['data']['iterations'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0]

fig,axes=plt.subplots(nrows=nrows,ncols=ncols,figsize=(ncols*6,nrows*4),sharey=True)
for i, experiment in enumerate([e for e in smc_results if e['proposal_mechanism'] != 'MH']):
    ax = axes[i // ncols, i % ncols]
    tree_len_mean = np.mean(experiment['tree_len'], axis=0)
    tree_len_std = np.std(experiment['tree_len'], axis=0)
    time_mean = np.mean(experiment['time'], axis=0)
    ax.plot(experiment['data']['iterations'], tree_len_mean, label='Mean Tree Length')
    ax.fill_between(experiment['data']['iterations'], tree_len_mean - tree_len_std, tree_len_mean + tree_len_std, alpha=0.3, label='Std Dev')
    ax.plot(mh_iterations, mh_tree_len_mean, label='MH Mean Tree Length', linestyle='--')
    ax.fill_between(mh_iterations, mh_tree_len_mean - mh_tree_len_std, mh_tree_len_mean + mh_tree_len_std, alpha=0.3, label='MH Std Dev', linestyle='--')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Wall Clock Time (s)')
    ax.set_ylabel('Tree Length')
    # set xlim to be from 0 to the first iteration where tree_len == 50
    ax.set_xlim([-1,10])
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Tree Length for SMC Experiments", fontsize=16,y=1.03)
plt.show()

In [ ]:
# plot mean and std of ess for all MCMC experiments, each plot having a different proposal mechanism.
num_plots = len(mcmc_results)-1
ncols = 5
nrows = int(np.ceil(num_plots / ncols))
mh_tree_len_mean = np.mean([experiment['tree_len'] for experiment in mcmc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_tree_len_std = np.std([experiment['tree_len'] for experiment in mcmc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
# MH ran for longer than the rest, so it is plotted against its own iterations.
mh_iterations = [experiment['data']['iterations'] for experiment in mcmc_results if experiment['proposal_mechanism'] == 'MH'][0]

fig,axes=plt.subplots(nrows=nrows,ncols=ncols,figsize=(ncols*6,nrows*4),sharey=True)
for i, experiment in enumerate([e for e in mcmc_results if e['proposal_mechanism'] != 'MH']):
    ax = axes[i // ncols, i % ncols]
    tree_len_mean = np.mean(experiment['tree_len'], axis=0)
    tree_len_std = np.std(experiment['tree_len'], axis=0)
    ax.plot(experiment['data']['iterations'], tree_len_mean, label='Mean Tree Length')
    ax.fill_between(experiment['data']['iterations'], tree_len_mean - tree_len_std, tree_len_mean + tree_len_std, alpha=0.3, label='Std Dev')
    ax.plot(mh_iterations, mh_tree_len_mean, label='MH Mean Tree Length', linestyle='--')
    ax.fill_between(mh_iterations, mh_tree_len_mean - mh_tree_len_std, mh_tree_len_mean + mh_tree_len_std, alpha=0.3, label='MH Std Dev', linestyle='--')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Tree Length')
    # set xlim to be from 0 to the first iteration where tree_len == 50
    # ax.set_xlim([-10,3000])
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Tree Length for MCMC Experiments", fontsize=16,y=1.03)
plt.show()

In [ ]:
# plot mean and std of ess for all SMC experiments, each plot having a different proposal mechanism.
num_plots = len(smc_results)-1
ncols = 5
nrows = int(np.ceil(num_plots / ncols))
mh_ess_mean = np.mean([experiment['ess'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_ess_std = np.std([experiment['ess'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
# MH was evaluated every 80th step of a longer run: plot it at those steps.
mh_iterations = [experiment['data']['iterations'] for experiment in smc_results if experiment['proposal_mechanism'] == 'MH'][0]

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 6, nrows * 4),sharey=True)
for i, experiment in enumerate([e for e in smc_results if e['proposal_mechanism'] != 'MH']):
    ax = axes[i // ncols, i % ncols]
    ess_mean = np.mean(experiment['ess'], axis=0)
    ess_std = np.std(experiment['ess'], axis=0)
    ax.plot(experiment['data']['iterations'], ess_mean, label='Mean ESS',linewidth=1)
    ax.fill_between(experiment['data']['iterations'], [max(0, a) for a in ess_mean - ess_std ], [min(40, a) for a in ess_mean + ess_std ], alpha=0.3, label='Std Dev')
    ax.plot(mh_iterations, mh_ess_mean, label='MH Mean ESS',linewidth=1)
    ax.fill_between(mh_iterations, [max(0, a) for a in mh_ess_mean - mh_ess_std], [min(40, a) for a in mh_ess_mean + mh_ess_std ], alpha=0.3, label='MH Std Dev')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Effective Sample Size (ESS)')
    ax.axhline(y=4, color='red', linestyle='--', label=f'N/2', linewidth=2)
    # ax.set_xlim([0, max(experiment['data']['iterations'])])
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Effective Sample Size (ESS) for SMC Experiments - {timestamp}", fontsize=16,y=1.035)
plt.show()

In [ ]:
# plot mean and std of ess for all MCMC experiments, each plot having a different proposal mechanism.
num_plots = len(mcmc_results)-1
ncols = 3
nrows = int(np.ceil(num_plots / ncols))
mh_tree_len_mean = np.mean([experiment['tree_len'] for experiment in mcmc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
mh_tree_len_std = np.std([experiment['tree_len'] for experiment in mcmc_results if experiment['proposal_mechanism'] == 'MH'][0], axis=0)
# MH ran for longer than the rest, so it is plotted against its own iterations.
mh_iterations = [experiment['data']['iterations'] for experiment in mcmc_results if experiment['proposal_mechanism'] == 'MH'][0]

fig,axes=plt.subplots(nrows=nrows,ncols=ncols,figsize=(ncols*6,nrows*4),sharey=True)
for i, experiment in enumerate([e for e in mcmc_results if e['proposal_mechanism'] != 'MH']):
    ax = axes[i // ncols, i % ncols]
    tree_len_mean = np.mean(experiment['tree_len'], axis=0)
    tree_len_std = np.std(experiment['tree_len'], axis=0)
    ax.plot(experiment['data']['iterations'], tree_len_mean, label='Mean Tree Length')
    ax.fill_between(experiment['data']['iterations'], tree_len_mean - tree_len_std, tree_len_mean + tree_len_std, alpha=0.3, label='Std Dev')
    ax.plot(mh_iterations, mh_tree_len_mean, label='MH Mean Tree Length', linestyle='--')
    ax.fill_between(mh_iterations, mh_tree_len_mean - mh_tree_len_std, mh_tree_len_mean + mh_tree_len_std, alpha=0.3, label='MH Std Dev', linestyle='--')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Tree Length')
    # set xlim to be from 0 to the first iteration where tree_len == 50
    # ax.set_xlim([-10,3000])
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Tree Length for MCMC Experiments", fontsize=16,y=1.03)
plt.show()

In [ ]:
# # Replace MH in smc_results and mcmc_results with the long MH results
# # 
# # drop subsample results where ss_prop = 0.5, 0.125 for DA, ss_prop = 2, 8 for Flat HINTS, and ss_prop = [1, 2], [1, 2, 4, 8] for HINTS
# smc_combined = [e for e in smc_results if not ((e['ss_prop'] in [0.5, 0.125, 2, 8, [1, 2], [1, 2, 4, 8]]))]
# mcmc_combined = [e for e in mcmc_results if not ((e['ss_prop'] in [0.5, 0.125, 2, 8, [1, 2], [1, 2, 4, 8]]))]

In [ ]:


# reverse order of DA experiments so it matches Flat HINTS and HINTS, which are ordered by ss_prop ascending
def reorder_results(results):
    # reverse DA 
    order = ["DA", "Flat HINTS", "HINTS", "MH"]
    ss_order = {proposal: sorted([expe for expe in results if expe['proposal_mechanism'] == proposal], key=lambda x: x['ss_prop'] if x['ss_prop'] is not None else float('inf')) for proposal in order}
    ss_reversed = {proposal: list(reversed(ss_order[proposal])) for proposal in order if proposal == "DA"}
    for proposal in order:
        if proposal == "DA":
            results = [expe for expe in results if expe['proposal_mechanism'] != proposal] + ss_reversed[proposal]
        else:
            results = [expe for expe in results if expe['proposal_mechanism'] != proposal] + ss_order[proposal]
    return results
smc_combined = reorder_results(smc_results)
mcmc_combined = reorder_results(mcmc_results)

In [ ]:
line_colours = {
    'MH':    ['#009E73'],
    'DA':    ['#8B0A2E', '#D55E00', '#E69F00', '#F0E442'],
    'HINTS2': ['#56B4E9', '#0072B2', '#2B2B7F', '#7B3294']}
shade_colours = {
    'MH':    ['#99D8C7'],
    'DA':    ['#E3B5C1', '#F2BF99', '#F5D999', '#FAF4B3'],
    'HINTS2': ['#BBE1F6', '#99C7E0', '#B9B9DC', '#C4A3CF']}

line_colours = {
    'MH':    ['#009E73'],
    # 'DA':    ['#CC79A7', '#0072B2', '#F0E442', '#56B4E9', '#E69F00'],
    'DA':    ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2'],
    'HINTS': ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2'],
    'Flat HINTS': ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2']
}
shade_colours = {
    'MH':    ['#99D8C7'],
    # 'DA':    ['#D9A3D9', '#99C7E0', '#FAF4B3', '#BBE1F6', '#F2BF99'],
    'DA':   ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0'],
    'HINTS': ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0'],
    'Flat HINTS': ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0']
}
X_LABELS = {"iteration": "Iteration", "time": "Sampling time (s)"}


def colour_index(experiment, results):
    """Rank of this experiment's ss_prop among its proposal's ss_props, so each
    subsample size keeps the same colour in every plot."""
    proposal = experiment['proposal_mechanism']
    ss_values = sorted({e['ss_prop'] for e in results if e['proposal_mechanism'] == proposal},
                       key=lambda s: -1 if s is None else s)
    return ss_values.index(experiment['ss_prop']) % len(line_colours[proposal])

# change colour_index to work with ss_prop being a list for HINTS2 experiments, so that the colour is determined by the last element of the list,

def colour_index(experiment, results):
    """Rank of this experiment's ss_prop among its proposal's ss_props, so each
    subsample size keeps the same colour in every plot."""
    proposal = experiment['proposal_mechanism']
    if isinstance(experiment['ss_prop'], list):
        ss_value = experiment['ss_prop'][-1]
    else:
        ss_value = experiment['ss_prop']
    ss_values = sorted({e['ss_prop'][-1] if isinstance(e['ss_prop'], list) else e['ss_prop'] for e in results if e['proposal_mechanism'] == proposal},
                       key=lambda s: -1 if s is None else s)
    return ss_values.index(ss_value) % len(line_colours[proposal])


def curve(experiment, key, x="iteration", n_grid=500):
    """(x values, mean, std) of a metric across runs, against iteration or,
    with x="time", against cumulative sampling time in seconds."""
    values = experiment['data'][key]                      # (n_runs, n_points)
    if x == "iteration":
        return experiment['data']['iterations'], values.mean(axis=0), values.std(axis=0)
    # Runs finish at different wallclock times, so they are averaged on a shared
    # time grid that stops when the fastest run finished; at each grid time a
    # run holds the value of the last iteration it had completed.
    times = experiment['time']
    grid = np.linspace(0.0, times[:, -1].min(), n_grid)
    at_grid = np.stack([v[np.clip(np.searchsorted(t, grid, side="right") - 1, 0, None)]
                        for t, v in zip(times, values)])
    return grid, at_grid.mean(axis=0), at_grid.std(axis=0)


def plot_metric(results, key, title, ylabel, x="iteration", xlim=None):
    plt.figure(figsize=(8, 6))
    for experiment in results:
        proposal = experiment['proposal_mechanism']
        k = colour_index(experiment, results)
        xs, mean, std = curve(experiment, key, x)
        label = proposal if experiment['ss_prop'] is None else f"{proposal} ss={experiment['ss_prop']}"
        plt.fill_between(xs, mean - std, mean + std, color=shade_colours[proposal][k], alpha=0.5, zorder=1)
        plt.plot(xs, mean, label=label, color=line_colours[proposal][k], zorder=2)
    plt.title(title)
    plt.xlabel(X_LABELS[x])
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()
    if xlim:
        plt.xlim(xlim)

# same as above but separate HINTS2 and DA into 2 subplots with MH on both
def plot_metric_split(results, key, title, samplername, ylabel, cols=3, x="iteration",footnote=None, xlim=None):
    fig, axes = plt.subplots(1, cols, figsize=(20, 6), sharey=True, sharex=True)
    for experiment in results:
        proposal = experiment['proposal_mechanism']
        k = colour_index(experiment, results)
        xs, mean, std = curve(experiment, key, x)
        if experiment['proposal_mechanism'] == "HINTS":
            label = f"HINTS (Hierarchy={experiment['ss_prop']})"
        elif experiment['proposal_mechanism'] == "Flat HINTS":
            label = f"Flat HINTS (Hierarchy={experiment['ss_prop']})"
        elif experiment['proposal_mechanism'] == "DA":
            label = f"DA (ss={experiment['ss_prop']})"
        else:
            label = proposal
        # label = proposal if experiment['ss_prop'] is None else f"ss={experiment['ss_prop']}"
        if proposal == "DA":
            ax = axes[0]
        elif proposal == "Flat HINTS":
            ax = axes[1]
        elif proposal == "HINTS":
            ax = axes[2]
        elif proposal == "MH":
            for ax in axes:
                ax.fill_between(xs, mean - std, mean + std, color=shade_colours[proposal][k], alpha=0.5, zorder=1)
                ax.plot(xs, mean, label=label, color=line_colours[proposal][k], zorder=2)
        if proposal in ["DA","Flat HINTS","HINTS"]:
            ax.fill_between(xs, mean - std, mean + std, color=shade_colours[proposal][k], alpha=0.5, zorder=1)
            ax.plot(xs, mean, label=label, color=line_colours[proposal][k], zorder=2)
            ax.set_title(f"{proposal}")
            ax.set_xlabel(X_LABELS[x])
        if xlim:
            ax.set_xlim(xlim)
        ax.set_ylabel(ylabel)
        ax.legend()
    if xlim:
        fig.suptitle(f"{samplername} {title} by {x} on {experiment['dataset']} for {experiment['data']['n_runs']} runs (First {max(xlim)} {"seconds" if x=="time" else "iterations"})")
    else:
        fig.suptitle(f"{samplername} {title} by {x} on {experiment['dataset']} for {experiment['data']['n_runs']} runs")
    # add footnote if provided
    if footnote:
        plt.figtext(0.5, -0.05, footnote, wrap=True, horizontalalignment='center', fontsize=10, fontstyle='italic')
    plt.show()


def mh_footnote(results, unit):
    """How MH's run differs from the rest, read off the files themselves."""
    mh = [e for e in results if e['proposal_mechanism'] == 'MH'][0]
    rest = [e for e in results if e['proposal_mechanism'] == 'HINTS'][0]
    note = (f"MH and DA were run for {mh['n_iters'] / rest['n_iters']:g} times as many "
            f"{unit}s as the other proposal mechanisms")
    if mh['stride'] > 1:
        note += (f", with its metrics evaluated every {mh['stride']}nd {unit} "
                 f"and plotted at those {unit}s")
    return note + "."

In [ ]:
footnote = mh_footnote(smc_combined, "step")
plot_metric_split(smc_combined, 'train_accuracy', "Training Accuracy", "SMC", "Accuracy",footnote=footnote)
plot_metric_split(smc_combined, 'train_accuracy', "Training Accuracy", "SMC", "Accuracy",x="time",footnote=footnote)

In [ ]:
plot_metric_split(smc_combined, 'test_accuracy', "Test Accuracy", "SMC", "Accuracy",footnote=footnote, xlim=[-1,50])
plot_metric_split(smc_combined, 'test_accuracy', "Test Accuracy", "SMC", "Accuracy", x="time",footnote=footnote, xlim=[-1,80])


In [ ]:
plot_metric_split(smc_combined, 'test_brier', "Test Brier Score", "SMC", "Brier Score",footnote=footnote)
plot_metric_split(smc_combined, 'test_brier', "Test Brier Score", "SMC", "Brier Score", x="time",footnote=footnote)

In [ ]:
plot_metric_split(smc_combined, 'test_macro_f1', "Test Macro F1", "SMC", "Macro F1 Score",footnote=footnote)
plot_metric_split(smc_combined, 'test_macro_f1', "Test Macro F1", "SMC", "Macro F1 Score", x="time",footnote=footnote)


In [ ]:
plot_metric_split(smc_combined, 'test_log_loss', "Test Log Loss", "SMC", "Log Loss",footnote=footnote)
plot_metric_split(smc_combined, 'test_log_loss', "Test Log Loss", "SMC", "Log Loss", x="time",footnote=footnote)


In [ ]:
footnote = mh_footnote(mcmc_combined, "iteration")
plot_metric_split(mcmc_combined, 'train_accuracy', "Training Accuracy", "MCMC", "Accuracy",footnote=footnote)
plot_metric_split(mcmc_combined, 'train_accuracy', "Training Accuracy", "MCMC", "Accuracy", x="time",footnote=footnote, xlim=[-1,10])


In [ ]:
plot_metric_split(mcmc_combined, 'test_accuracy', "Test Accuracy", "MCMC", "Accuracy",footnote=footnote)
plot_metric_split(mcmc_combined, 'test_accuracy', "Test Accuracy", "MCMC", "Accuracy", x="time",footnote=footnote)


In [ ]:
plot_metric_split(mcmc_combined, 'test_log_loss', "Test Log Loss", "MCMC", "Log Loss",footnote=footnote)
plot_metric_split(mcmc_combined, 'test_log_loss', "Test Log Loss", "MCMC", "Log Loss", x="time",footnote=footnote, xlim=[0,20])


In [ ]:
# rearrange experiments in smc_combined and mcmc_combined so that MH is last
smc_combined = sorted(smc_combined, key=lambda x: (x['proposal_mechanism'] == 'MH', x['proposal_mechanism'], x['ss_prop']))
mcmc_combined = sorted(mcmc_combined, key=lambda x: (x['proposal_mechanism'] == 'MH', x['proposal_mechanism'], x['ss_prop']))

In [ ]:
# From the h5 files, the fate of every proposed move, broken down by move type.
# Each Run_N holds "move" (0-3, MOVES) and "outcome" (0-5, OUTCOMES).
from matplotlib.patches import Patch


MOVES = ("stay", "grow", "prune", "change")
MOVE_CODE = {name: i for i, name in enumerate(MOVES)}

STAY = 0            # no valid move was available
BARRED = 1          # the proposal correction was BARRED (invalid split, or a reverse path barred at max_tree_size)
SCREENED_OUT = 2    # rejected by the surrogate on a subsample (DA, HINTS)
INADMISSIBLE = 3    # rejected by a min_samples_leaf admissibility check
PROPOSED = 4        # survived the proposal, handed to the outer MH step
APPLIED = 5         # written into a HINTS sweep's working tree
OUTCOMES = ("stay", "barred", "screened_out", "inadmissible", "proposed",
            "applied")

# Segments stack in outcome order, so the hues were picked to clear the
# adjacent-pair colour-blindness and contrast checks in that order -- the
# saturated red/yellow/green they replace failed both, and pure yellow on a
# white ground was invisible.
OUTCOME_COLOURS = {
    STAY:         "#8c8578",  # warm grey -- the null outcome
    BARRED:       "#e27860",  # orange-red
    SCREENED_OUT: "#a59ae6",  # violet
    INADMISSIBLE: "#e9de65",  # amber
    PROPOSED:     "#8bb4e5",  # blue
    APPLIED:      "#6AE26A",  # green
}

SURFACE = "#ffffff"
BAR_W = 0.5
INSIDE_MIN = 0.055   # segments thinner than this fraction of the axis get a lifted label
GAP = 0.052          # vertical spacing of the lifted labels, same units


def move_outcome_counts(experiment):
    """(len(MOVES), len(OUTCOMES)) counts, pooled over every chain of a run."""
    h5_path = experiment['h5_path']
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")),
                      key=lambda k: int(k[len("Run_"):]))
        moves = np.concatenate([f[run]["move"][()] for run in runs])
        outcomes = np.concatenate([f[run]["outcome"][()] for run in runs])
    flat = moves.astype(np.int64) * len(OUTCOMES) + outcomes.astype(np.int64)
    return np.bincount(flat, minlength=len(MOVES) * len(OUTCOMES)).reshape(
        len(MOVES), len(OUTCOMES))


def _si(n):
    for cut, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if n >= cut:
            return f"{n / cut:.1f}{suffix}".replace(".0", "")
    return str(int(n))


def _pct(value):
    return f"{value:.2f}%" if value >= 1 else f"{value:.3f}%"


def _n_lifted(heights, span):
    """How many segments are too thin to hold their own label."""
    return sum(1 for h in heights if 0 < h < INSIDE_MIN * span)


def _label_bar(ax, i, heights, shares, span):
    """Percentages beside the segments they belong to, each in that outcome's
    colour over a black outline -- the outline is what keeps a label legible
    where it sits on its own fill, or on the white ground above the bar."""
    # A thin outline: a heavy one swallows the glyph at this size, which is
    # what makes a label unreadable when it sits on a fill of its own colour.
    stroke = [pe.withStroke(linewidth=1.5, foreground="black")]
    gap = GAP * span
    y = heights.sum() + gap
    j = 0
    for k in range(len(OUTCOMES)):
        if heights[k] == 0:
            continue
        centre = heights[:k].sum() + heights[k] / 2
        if heights[k] >= INSIDE_MIN * span:
            ax.text(i, centre, _pct(shares[k]), ha='center', va='center',
                    color=OUTCOME_COLOURS[k], size=10, weight='bold',
                    zorder=4, path_effects=stroke)
        else:
            # Out of the segment sideways, then straight up to the label, so a
            # leader never cuts across the stack or a neighbouring bar.
            ax.annotate(_pct(shares[k]), xy=(i + BAR_W / 2, centre),
                        xytext=(i + BAR_W / 2 + 0.05 + 0.055 * j, y),
                        ha='left', va='center', size=10, weight='bold', zorder=4,
                        color=OUTCOME_COLOURS[k], path_effects=stroke,
                        annotation_clip=False,
                        arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=2,
                                        color=OUTCOME_COLOURS[k],
                                        connectionstyle="angle,angleA=0,angleB=90"))
            y += gap
            j += 1


def plot_move_outcomes(results, samplername, share=True, footnote=None):
    """One panel per experiment: proposal outcomes stacked by move type.

    share=True stacks every bar to 100% and prints its total underneath -- the
    totals run over orders of magnitude between moves and between experiments,
    so raw counts flatten every bar but the largest. share=False plots the
    counts themselves, each panel on its own y axis.
    """
    counts = [move_outcome_counts(experiment) for experiment in results]

    ncols = 3
    nrows = int(np.ceil(len(results) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.4, nrows * 4.4),
                             squeeze=False)
    axes = axes.ravel()
    xs = np.arange(len(MOVES))

    panels = []
    for experiment, count in zip(results, counts):
        totals = count.sum(axis=1)
        shares = np.divide(count * 100.0, totals[:, None], out=np.zeros(count.shape),
                           where=totals[:, None] > 0)
        heights = shares if share else count.astype(float)
        span = 100.0 if share else max(heights.sum(axis=1).max(), 1.0)
        # Headroom for the tallest column of lifted labels, so none is drawn
        # outside the axes -- tight_layout would squash the panel to fit it.
        n_lift = max(_n_lifted(heights[i], span) for i in range(len(MOVES)))
        panels.append((experiment, totals, shares, heights, span,
                       span * (1 + GAP * (n_lift + 0.8))))

    # A shared limit only means anything when every panel is on the same 0-100 scale.
    shared_top = max(p[-1] for p in panels) if share else None

    for ax, (experiment, totals, shares, heights, span, top) in zip(axes, panels):
        bottom = np.zeros(len(MOVES))
        for k in range(len(OUTCOMES)):
            ax.bar(xs, heights[:, k], width=BAR_W, bottom=bottom, zorder=2,
                   color=OUTCOME_COLOURS[k], edgecolor=SURFACE, linewidth=1.5)
            bottom += heights[:, k]

        for i in range(len(MOVES)):
            if totals[i] > 0:
                _label_bar(ax, i, heights[i], shares[i], span)

        ss = experiment['ss_prop']
        if experiment['proposal_mechanism'] == "HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "Flat HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "DA":
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        else:
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        # Under each move: how many proposals it drew, and what share of all
        # the run's proposals that is -- the stack above is a share of n alone.
        proposed_share = 100.0 * totals / max(totals.sum(), 1)
        ax.set_xticks(xs, [f"{m}\nn={_si(t)}\n{_pct(s)}"
                           for m, t, s in zip(MOVES, totals, proposed_share)])
        ax.set_xlim(-0.55, len(MOVES) - 0.15)
        ax.set_ylim(0, shared_top if share else top)
        ax.set_xlabel('Move type')
        ax.set_ylabel('Share of proposals (%)' if share else 'Proposals')
        ax.grid(axis='x', visible=False)
        if share:
            ax.set_yticks(np.arange(0, 101, 25))

    for ax in axes[len(results):]:      # blank out the unfilled grid slots
        ax.set_visible(False)

    fig.legend(handles=[Patch(facecolor=OUTCOME_COLOURS[k], label=OUTCOMES[k])
                        for k in range(len(OUTCOMES))],
               loc='upper center', ncol=len(OUTCOMES), frameon=False,
               bbox_to_anchor=(0.5, 0.97))
    fig.suptitle(f"Proposal outcome by move type -- {samplername} experiments, {timestamp}",
                 fontsize=16)
    if footnote:
        fig.text(0.5, -0.01, footnote, ha='center', fontsize=10, fontstyle='italic')
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    plt.show()

# smc_combined = [exp for exp in smc_results if exp['proposal_mechanism'] != "MH"] + long_smc_mh
# mcmc_combined = [exp for exp in mcmc_results if exp['proposal_mechanism'] != "MH"] + long_mcmc_mh
plot_move_outcomes(smc_combined, "SMC", share=False)
plot_move_outcomes(mcmc_combined, "MCMC")

In [ ]:
# The screen crossed with the sampler's own accept/reject. The outcome column
# alone is only half the story: DA and HINTS screen a move against a surrogate
# before the outer Metropolis step ever sees it, so a move can pass the screen
# and still be thrown away -- or, inside a HINTS sweep, be thrown out by the
# screen while the sweep it belonged to lands. move_fates() does that join on
# the call index; it lives in the experiment scripts, so reuse it rather than
# restate it here.
import sys

_diag_dir = os.path.join(os.getcwd(), "examples", "incremental_decision_tree")
if _diag_dir not in sys.path:
    sys.path.insert(0, _diag_dir)
import plot_diagnostics as _pd

# _pd.FATES in order, under the names we read them by. The last two are not
# fates of a proposed move at all -- they never reached the target -- but they
# are most of the work on some move types, so dropping them would renormalise
# the bars onto a fraction of what the proposal actually did.
FATE_LABELS = ("Approx Accept + Accept", "MH Accept", "Approx Accept + Reject",
               "Approx Reject + Accept", "Approx Reject", "MH Reject",
               "barred / inadmissible", "no move available")

FATE_COLOURS = {
    0: "#98df8a",  # light green -- passed the screen and the sampler took it
    1: "#22A822",  # dark green    -- accepted with no screen (MH, DA's skip path)
    2: "#ff7f0e",  # orange  -- screen passed it, the sampler did not
    3: "#54c8e2",  # violet  -- HINTS: the sweep landed, this move did not
    4: "#ff9896",  # amber   -- the surrogate threw it out
    5: "#e34948",  # red     -- rejected, never screened
    6: "#8c8578",  # warm grey
    7: "#c7c2b8",  # light grey
}


# The coarse view: what actually became of the move. "Accept" is _pd's own
# acceptance numerator, so a move screened out of a HINTS sweep that landed
# counts as a Reject -- the sweep was accepted, that move was not.
FATE_GROUPS = (
    ("Accept",      "#22A822", _pd.ACCEPTED_FATES),
    ("Reject",      "#e34948", tuple(k for k in range(6)
                                     if k not in _pd.ACCEPTED_FATES)),
    ("Barred",      "#8c8578", (6,)),
    ("unavailable", "#c7c2b8", (7,)),
)


def group_fates(count):
    """Collapse a (len(MOVES), len(FATE_LABELS)) table onto FATE_GROUPS."""
    return np.stack([count[:, list(cols)].sum(axis=1)
                     for _, _, cols in FATE_GROUPS], axis=1)


def move_fate_counts(experiment):
    """(len(MOVES), len(FATE_LABELS)) counts, pooled over every chain."""
    h5_path = experiment['h5_path']
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")),
                      key=lambda k: int(k[len("Run_"):]))
        tables = [_pd.move_fates(f[run]) for run in runs]
    return sum(t for t in tables if t is not None)


def _label_bar_fate(ax, i, heights, shares, span, colours):
    """As _label_bar above, but over an arbitrary colour map."""
    stroke = [pe.withStroke(linewidth=1.4, foreground="black")]
    gap = GAP * span
    y = heights.sum() + gap
    j = 0
    for k in range(len(heights)):
        if heights[k] == 0:
            continue
        centre = heights[:k].sum() + heights[k] / 2
        if heights[k] >= INSIDE_MIN * span:
            ax.text(i, centre, _pct(shares[k]), ha='center', va='center',
                    color=colours[k], size=10.5, weight='bold', zorder=4,
                    path_effects=stroke)
        else:
            ax.annotate(_pct(shares[k]), xy=(i + BAR_W / 2, centre),
                        xytext=(i + BAR_W / 2 + 0.05 + 0.055 * j, y),
                        ha='left', va='center', size=9, weight='bold', zorder=4,
                        color=colours[k], path_effects=stroke,
                        annotation_clip=False,
                        arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=2,
                                        color=colours[k],
                                        connectionstyle="angle,angleA=0,angleB=90"))
            y += gap
            j += 1


def plot_move_fates(results, samplername, share=True, footnote=None, ncols=3,
                    grouped=False):
    """One panel per experiment: where each move type's proposals ended up.

    Same layout as plot_move_outcomes, over the screen-crossed-with-accept
    classification instead of the proposal-internal outcome.

    grouped=True collapses the six fates onto Accept / Reject, keeping barred
    and unavailable separate -- neither ever reached the outer step, so folding
    them into Reject would read as the sampler having turned them down.
    """
    counts = [move_fate_counts(experiment) for experiment in results]
    if grouped:
        counts = [group_fates(count) for count in counts]
        labels = [name for name, _, _ in FATE_GROUPS]
        colours = {k: hexcode for k, (_, hexcode, _) in enumerate(FATE_GROUPS)}
    else:
        labels, colours = list(FATE_LABELS), FATE_COLOURS
    n_fates = len(labels)

    nrows = int(np.ceil(len(results) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.4, nrows * 4.4),
                             squeeze=False)
    axes = axes.ravel()
    xs = np.arange(len(MOVES))

    panels = []
    for experiment, count in zip(results, counts):
        totals = count.sum(axis=1)
        shares = np.divide(count * 100.0, totals[:, None], out=np.zeros(count.shape),
                           where=totals[:, None] > 0)
        heights = shares if share else count.astype(float)
        span = 100.0 if share else max(heights.sum(axis=1).max(), 1.0)
        n_lift = max(sum(1 for h in heights[i] if 0 < h < INSIDE_MIN * span)
                     for i in range(len(MOVES)))
        panels.append((experiment, totals, shares, heights, span,
                       span * (1 + GAP * (n_lift + 0.8))))

    shared_top = max(p[-1] for p in panels) if share else None

    for ax, (experiment, totals, shares, heights, span, top) in zip(axes, panels):
        bottom = np.zeros(len(MOVES))
        for k in range(n_fates):
            ax.bar(xs, heights[:, k], width=BAR_W, bottom=bottom, zorder=2,
                   color=colours[k], edgecolor=SURFACE, linewidth=1.5)
            bottom += heights[:, k]

        for i in range(len(MOVES)):
            if totals[i] > 0:
                _label_bar_fate(ax, i, heights[i], shares[i], span, colours)

        ss = experiment['ss_prop']
        if experiment['proposal_mechanism'] == "HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "Flat HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "DA":
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        else:
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        move_share = 100.0 * totals / max(totals.sum(), 1)
        ax.set_xticks(xs, [f"{m}\n{_si(t)}\n{_pct(s)}"
                           for m, t, s in zip(MOVES, totals, move_share)])
        ax.set_xlim(-0.55, len(MOVES) - 0.15)
        ax.set_ylim(0, shared_top if share else top)
        ax.set_xlabel('Move type')
        ax.set_ylabel('Share of proposals (%)' if share else 'Proposals')
        ax.grid(axis='x', visible=False)
        if share:
            ax.set_yticks(np.arange(0, 101, 25))

    for ax in axes[len(results):]:
        ax.set_visible(False)

    # if not first column, remove y axis labels
    for ax in axes:
        if ax.get_subplotspec().colspan.start > 0:
            ax.set_ylabel('')
            ax.set_yticklabels([])

    fig.legend(handles=[Patch(facecolor=colours[k], label=labels[k])
                        for k in range(n_fates)],
               loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle(f"Move Outcomes over {experiment['data']['n_runs']} chains in {samplername} experiments on {experiment['dataset']}",
                 fontsize=20,y=1, fontweight='bold')
    if footnote:
        fig.text(0.5, -0.01, footnote, ha='center', fontsize=10, fontstyle='italic')
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    plt.show()


plot_move_fates(mcmc_combined, "MCMC")
plot_move_fates(mcmc_combined, "MCMC", grouped=True)
# SMC reweights particles instead of rejecting them, so every call that
# proposed something became the new particle: the two "Reject" fates are
# structurally empty there, and "Accept" means "survived the proposal".
plot_move_fates(smc_combined, "SMC",
                footnote="SMC has no accept/reject step -- it reweights, so nothing "
                         "lands in the Reject fates and Accept means the proposal survived.")
plot_move_fates(smc_combined, "SMC", grouped=True,
                footnote="SMC has no accept/reject step -- it reweights, so nothing "
                         "lands in Reject and Accept means the proposal survived.")

In [ ]:
plot_move_fates(mcmc_combined, "MCMC", grouped=True)
plot_move_fates(smc_combined, "SMC", grouped=True,
                footnote="SMC has no accept/reject step -- it reweights, so nothing "
                         "lands in Reject and Accept means the proposal survived.")


In [ ]:
group_fates(move_fate_counts(smc_combined[4]))

In [ ]:
import pandas as pd
# give me the raw count of each fate for each move type, for each experiment, in a table with percentages in brackets
def move_fate_table(results, samplername, grouped=False):
    counts = [move_fate_counts(experiment) for experiment in results]
    if grouped:
        counts = [group_fates(count) for count in counts]
        labels = [name for name, _, _ in FATE_GROUPS]
    else:
        labels = list(FATE_LABELS)
    n_fates = len(labels)

    table = []
    for experiment, count in zip(results, counts):
        totals = count.sum(axis=1)
        shares = np.divide(count , totals[:, None], out=np.zeros(count.shape),
                           where=totals[:, None] > 0)
        row = {
            "proposal_mechanism": experiment['proposal_mechanism']
        }
        if experiment['proposal_mechanism'] in ["HINTS", "Flat HINTS"]:
            row["proposal_mechanism"] = f"\\makecell{{{experiment['proposal_mechanism']} \\\\ ${experiment['ss_prop']}$}}"
        elif experiment['proposal_mechanism'] == "DA":
            row["proposal_mechanism"] = f"\\makecell{{{experiment['proposal_mechanism']} \\\\ {float(experiment['ss_prop']):.3g}}}"
        else:
            row["proposal_mechanism"] = "\\makecell{MH}"
        for i, move in enumerate(MOVES):
            for j, fate in enumerate(labels):
                if count[i, j] > 0:
                    # add latex newline between count and percentage
                    row[f"{move}_{fate}"] = f"\\makecell{{{count[i, j]:,} \\\\ ({shares[i, j]:.2g})}}"
        table.append(row)
    return table

mcmc_counts = move_fate_table(mcmc_combined, "MCMC", grouped=True)
smc_counts = move_fate_table(smc_combined, "SMC", grouped=True)

In [ ]:
pd.DataFrame(mcmc_counts)

In [ ]:
# create latex tables with the outputs and have the highest percentage in each column bolded
# have the {move}_{fate} split up so that {move} is a heading with each {fate} as a subheading, and the values in the table are the counts with percentages in brackets
# embolden the highest percentage in each column and have the percentage appear below the count in the table, so that the table is more readable
def create_latex_table(counts, samplername):
    df = pd.DataFrame(counts)
    df = df.fillna("--")
    # split the columns into multiindex
    new_columns = []
    for col in df.columns:
        if col in ["proposal_mechanism", "ss_prop"]:
            new_columns.append((col, ""))
        else:
            move, fate = col.split("_")
            new_columns.append((move, fate))
    df.columns = pd.MultiIndex.from_tuples(new_columns)
    # bold the highest percentage in each column
    for move in MOVES:
        for fate in FATE_LABELS:
            col = (move, fate)
            if col in df.columns:
                percentages = df[col].str.extract(r"\(([\d\.]+)\)").astype(float)
                max_idx = percentages.idxmax()[0]
                df.loc[max_idx, col] = "\\textbf{" + df.loc[max_idx, col] + "}"
    latex_table = df.to_latex(index=False, escape=False, multicolumn=True, multicolumn_format='c', column_format='l' + 'c' * (len(df.columns) - 2), multirow=False)
    return latex_table

print(create_latex_table(mcmc_counts, "MCMC"))